# Multi-Agent Supervisor-Worker Pattern Example

This notebook demonstrates the supervisor-worker pattern from Chapter 4.

## Overview
We'll build a market research system with:
- **News Agent**: Researches recent news and developments
- **Financial Agent**: Analyzes financial metrics and performance
- **Sentiment Agent**: Analyzes market sentiment from social media
- **Supervisor Agent**: Coordinates the workers and synthesizes results

## Setup

First, install the required packages:

In [ ]:
# Uncomment to install
# !pip install strands-agents strands-agents-tools yfinance alpha-vantage

In [ ]:
import os
import yfinance as yf
from alpha_vantage.timeseries import TimeSeries
from strands import Agent
from strands.tool import tool
from strands_tools.tavily import tavily_search
from strands_tools import http_request

## Configure API Keys

Set up your API keys for web search and financial data access.

In [ ]:
# Set your API keys
# Option 1: Set as environment variables (recommended)
# os.environ['TAVILY_API_KEY'] = 'your-tavily-api-key-here'
# os.environ['ALPHA_VANTAGE_API_KEY'] = 'your-alpha-vantage-api-key-here'

# Option 2: Or set them in your shell before running Jupyter
# export TAVILY_API_KEY='your-tavily-api-key-here'
# export ALPHA_VANTAGE_API_KEY='your-alpha-vantage-api-key-here'

# Get free API keys:
# - Tavily: https://www.tavily.com/
# - Alpha Vantage: https://www.alphavantage.co/support/#api-key

# Verify API keys are set
if not os.getenv('TAVILY_API_KEY'):
    print("⚠️  Warning: TAVILY_API_KEY not set. Web search will not work.")
else:
    print("✓ TAVILY_API_KEY is configured")

if not os.getenv('ALPHA_VANTAGE_API_KEY'):
    print("⚠️  Warning: ALPHA_VANTAGE_API_KEY not set. Using yfinance (free, no key needed) for financial data.")
else:
    print("✓ ALPHA_VANTAGE_API_KEY is configured")

## Create Custom Financial Tools

We'll create custom tools that wrap financial APIs for our agents to use.

In [ ]:
@tool
def get_stock_price(symbol: str) -> str:
    """
    Get current stock price and key financial metrics using Yahoo Finance.
    
    Args:
        symbol: Stock ticker symbol (e.g., 'AAPL', 'TSLA', 'MSFT')
    
    Returns:
        String with current price, market cap, P/E ratio, and other key metrics
    """
    try:
        stock = yf.Ticker(symbol)
        info = stock.info
        
        result = f"""Stock: {symbol}
Current Price: ${info.get('currentPrice', 'N/A')}
Market Cap: ${info.get('marketCap', 'N/A'):,}
P/E Ratio: {info.get('trailingPE', 'N/A')}
52 Week High: ${info.get('fiftyTwoWeekHigh', 'N/A')}
52 Week Low: ${info.get('fiftyTwoWeekLow', 'N/A')}
Average Volume: {info.get('averageVolume', 'N/A'):,}
Sector: {info.get('sector', 'N/A')}
Industry: {info.get('industry', 'N/A')}"""
        return result
    except Exception as e:
        return f"Error fetching stock data for {symbol}: {str(e)}"

@tool
def get_stock_history(symbol: str, period: str = "1mo") -> str:
    """
    Get historical stock price data.
    
    Args:
        symbol: Stock ticker symbol (e.g., 'AAPL', 'TSLA')
        period: Time period - valid values: 1d, 5d, 1mo, 3mo, 6mo, 1y, 2y, 5y, 10y, ytd, max
    
    Returns:
        String with historical price summary and recent trends
    """
    try:
        stock = yf.Ticker(symbol)
        hist = stock.history(period=period)
        
        if hist.empty:
            return f"No historical data available for {symbol}"
        
        start_price = hist['Close'].iloc[0]
        end_price = hist['Close'].iloc[-1]
        change_pct = ((end_price - start_price) / start_price) * 100
        
        result = f"""Historical Data for {symbol} ({period}):
Starting Price: ${start_price:.2f}
Ending Price: ${end_price:.2f}
Change: {change_pct:+.2f}%
Highest: ${hist['High'].max():.2f}
Lowest: ${hist['Low'].min():.2f}
Average Volume: {hist['Volume'].mean():,.0f}

Recent 5-day trend:
{hist[['Close', 'Volume']].tail().to_string()}"""
        return result
    except Exception as e:
        return f"Error fetching historical data for {symbol}: {str(e)}"

@tool
def get_company_financials(symbol: str) -> str:
    """
    Get company financial statements and key metrics.
    
    Args:
        symbol: Stock ticker symbol (e.g., 'AAPL', 'TSLA')
    
    Returns:
        String with revenue, earnings, profit margins, and growth metrics
    """
    try:
        stock = yf.Ticker(symbol)
        info = stock.info
        
        result = f"""Financial Metrics for {symbol}:
Revenue: ${info.get('totalRevenue', 'N/A'):,}
Revenue Growth: {info.get('revenueGrowth', 'N/A')}
Gross Profit: ${info.get('grossProfits', 'N/A'):,}
Profit Margin: {info.get('profitMargins', 'N/A')}
Operating Margin: {info.get('operatingMargins', 'N/A')}
EBITDA: ${info.get('ebitda', 'N/A'):,}
Earnings Growth: {info.get('earningsGrowth', 'N/A')}
Return on Equity: {info.get('returnOnEquity', 'N/A')}
Debt to Equity: {info.get('debtToEquity', 'N/A')}
Free Cash Flow: ${info.get('freeCashflow', 'N/A'):,}"""
        return result
    except Exception as e:
        return f"Error fetching financial data for {symbol}: {str(e)}"

print("✓ Custom financial tools created")

## Alternative Financial APIs

The example above uses **yfinance** (free, no API key). You can also use:

### Alpha Vantage (Free tier available)
```python
from alpha_vantage.timeseries import TimeSeries

@tool
def get_alpha_vantage_data(symbol: str) -> str:
    """Get stock data from Alpha Vantage API"""
    ts = TimeSeries(key=os.getenv('ALPHA_VANTAGE_API_KEY'), output_format='pandas')
    data, meta_data = ts.get_quote_endpoint(symbol=symbol)
    return data.to_string()
```

### Other Options
- **Polygon.io** - Real-time market data
- **IEX Cloud** - Financial data and insights  
- **Finnhub** - Stock market data and news
- **Financial Modeling Prep** - Company financials

See [AWS Financial Analysis Agent](https://aws.amazon.com/blogs/machine-learning/build-an-intelligent-financial-analysis-agent-with-langgraph-and-strands-agents/) for production examples.

## Create Worker Agents with Real Tools

Each worker agent specializes in a specific domain and uses real APIs.

In [ ]:
# News Agent - Uses Tavily for real-time web search
news_agent = Agent(
    name="news_agent",
    instructions="""You are a news research specialist. Search for recent news and developments about companies and industries.
    
    When searching:
    - Focus on recent news (use time_range='d' for daily, 'w' for weekly)
    - Look for reliable sources like major news outlets
    - Summarize key developments and trends
    - Include specific facts, dates, and figures when available
    """,
    tools=[tavily_search]
)

print("✓ News Agent created with Tavily search")

In [ ]:
# Financial Agent - Uses real financial APIs
financial_agent = Agent(
    name="financial_agent",
    instructions="""You are a financial analysis specialist. Analyze financial metrics, stock performance, and company valuations.
    
    When analyzing:
    - Use get_stock_price() to get current prices and key metrics
    - Use get_stock_history() to analyze price trends and performance
    - Use get_company_financials() to get revenue, earnings, and growth data
    - Compare companies within the same sector
    - Provide data-driven insights with specific numbers
    - Calculate growth rates and compare to industry averages
    """,
    tools=[get_stock_price, get_stock_history, get_company_financials, tavily_search]
)

print("✓ Financial Agent created with real financial APIs (yfinance)")

In [ ]:
# Sentiment Agent - Uses web search to analyze market sentiment
sentiment_agent = Agent(
    name="sentiment_agent",
    instructions="""You are a market sentiment analyst. Analyze market sentiment from social media, forums, and investor discussions.
    
    When analyzing sentiment:
    - Search for discussions on Reddit, Twitter, and investment forums
    - Look for both bullish and bearish opinions
    - Identify common concerns and excitement points
    - Distinguish between retail and institutional investor sentiment
    - Summarize the overall market mood
    """,
    tools=[tavily_search]
)

print("✓ Sentiment Agent created with web search")

## Create Supervisor Agent

The supervisor coordinates the worker agents and synthesizes their results.

In [ ]:
supervisor = Agent(
    name="research_supervisor",
    instructions="""You are a market research coordinator. When asked about investment opportunities:
    
    1. Analyze the question to determine what information is needed
    2. Delegate to specialist agents:
       - news_agent: For recent developments and industry news
       - financial_agent: For financial metrics and performance data
       - sentiment_agent: For market sentiment and investor opinions
    3. Synthesize their responses into a coherent investment recommendation
    
    Always provide balanced analysis considering all perspectives.""",
    tools=[
        news_agent.as_tool(),
        financial_agent.as_tool(),
        sentiment_agent.as_tool()
    ]
)

print("✓ Supervisor Agent created")

## Run the Multi-Agent System

Now let's ask the supervisor a question and watch it coordinate the worker agents.

In [ ]:
question = "Should I invest in electric vehicle companies?"

print(f"Question: {question}\n")
print("Processing...\n")

response = supervisor.invoke(question)

print("="*80)
print("SUPERVISOR RESPONSE:")
print("="*80)
print(response)

## Try Different Questions

Test the system with various investment questions:

In [ ]:
questions = [
    "What's the outlook for renewable energy stocks?",
    "Should I invest in AI companies right now?",
    "Is the semiconductor industry a good investment?"
]

for q in questions:
    print(f"\n{'='*80}")
    print(f"Question: {q}")
    print(f"{'='*80}")
    response = supervisor.invoke(q)
    print(response)
    print()

## Key Takeaways

1. **Real Financial APIs**: Uses Yahoo Finance (yfinance) for actual stock data - no API key needed
2. **Custom Tools**: Created specialized financial tools using @tool decorator
3. **Web Search Integration**: Tavily provides real-time news and market sentiment
4. **Specialization**: Each worker agent focuses on one domain with appropriate tools
5. **Coordination**: The supervisor delegates tasks and synthesizes results
6. **Modularity**: Easy to add new agents or swap APIs without changing architecture

## Next Steps

To make this production-ready:
- Add error handling and retry logic for API failures
- Implement caching to reduce API calls and costs
- Add rate limiting to respect API quotas
- Use async execution for parallel processing
- Add logging and monitoring with LangFuse or similar
- Consider adding more specialized agents:
  - Technical Analysis Agent (RSI, MACD, moving averages)
  - Regulatory Agent (SEC filings, compliance)
  - Options Agent (options chain analysis)
  - Portfolio Agent (diversification, risk assessment)

## Additional Resources

- [Strands Agents Documentation](https://strandsagents.com/)
- [Strands Tools GitHub](https://github.com/strands-agents/tools)
- [AWS Financial Analysis Agent Blog](https://aws.amazon.com/blogs/machine-learning/build-an-intelligent-financial-analysis-agent-with-langgraph-and-strands-agents/)
- [AWS Strands Course Repository](https://github.com/aws-samples/sample-getting-started-with-strands-agents-course)
- [yfinance Documentation](https://pypi.org/project/yfinance/)
- [Alpha Vantage API](https://www.alphavantage.co/documentation/)
- [Tavily API Documentation](https://www.tavily.com/)